# Colab smoke test (no zip, direct file download)

CSVファイルを直接作成し、そのままダウンロードします。Zip化は行いません。

In [ ]:
!pip -q install pandas

import pandas as pd
import os

games = pd.DataFrame({
    'match_id': ['demo-001', 'demo-002', 'demo-003', 'demo-004'],
    'datetime': pd.to_datetime(['2024-01-01', '2024-01-03', '2024-01-05', '2024-01-07'], utc=True),
    'home_team': ['A', 'B', 'A', 'C'],
    'away_team': ['B', 'A', 'C', 'A'],
    'home_score': [2, 1, 3, 0],
    'away_score': [1, 1, 0, 2],
    'source': ['smoke_test'] * 4,
})

games = games.sort_values(['datetime', 'match_id']).copy()
games['home_win_prior'] = games.groupby('home_team')['home_score'].transform(lambda s: s.shift().expanding().mean())
games['away_win_prior'] = games.groupby('away_team')['away_score'].transform(lambda s: s.shift().expanding().mean())
games['prediction_cutoff_at'] = games['datetime'] - pd.Timedelta(hours=1)

assert games['datetime'].is_monotonic_increasing
assert (pd.to_datetime(games['prediction_cutoff_at'], utc=True) < games['datetime']).all()
print('Chronology and cutoff checks: PASS')

csv_path = '/content/games_with_features.csv'
games.to_csv(csv_path, index=False)

print('File exists:', os.path.exists(csv_path))
print('File size (bytes):', os.path.getsize(csv_path))
print()
print(games)

from google.colab import files
files.download(csv_path)